#AiZynthFinder-Based Retrosynthesis Google Colab Workflow

This Google Colab notebook is designed to facilitate retrosynthetic planning using the AiZynthFinder architecture.
Note: This repository and the associated notebook are currently in active development for educational purposes. The code is provided as-is, and feedback or suggestions for improvement from the community are welcomed.

*For more information, visit the project GitHub repository*:
https://github.com/Sunil-Paliwal/aizynthfinder-retrosynthesis-colab


##Overview
This work is based on the open-source software AiZynthFinder, as described in the paper: Genheden et al., J. Cheminform (2020) 12:70. The original software utilizes a Monte Carlo tree search and a neural network policy to recursively break down molecules into purchasable precursors.

Paper link: https://doi.org/10.1186/s13321-020-00472-1

---

### **Quick Start**

To get started with the retrosynthesis workflow, follow these steps in order.

#### 1. Preparation
* First, ensure you are logged into your Google account.
* Open the notebook in Google Colab by clicking the link below:

  **[AiZynthFinder Notebook in Google Colab](https://colab.research.google.com/github/Sunil-Paliwal/aizynthfinder-retrosynthesis-colab/blob/main/AiZynthFinder_Retrosynthesis_Colab.ipynb)**
* Then, click **"Copy to Drive"** to save your own editable copy of the notebook.

#### 2. Setup & Run Cells
* **Cell #1**: Run this first by clicking the ▶ (play) button. It creates a separate Python 3.11 environment and installs `aizynthfinder` and its dependencies into it.
* **Cell #2**: Run this second by clicking the ▶ (play) button to download the public USPTO models and ZINC stock collection.
* **Cell #3**: Run this last by clicking the ▶ (play) button to launch the interactive search interface.

### 3. Perform Analysis
* Once the interface has appeared, you can enter a new SMILES string and click **"Run Search"** for each molecule.

> **Note:** Because the Python 3.11 environment and the downloaded models are stored in the temporary Colab session storage (`/content`), they are lost when the session disconnects or is closed. After a disconnect, simply re-run all three cells (#1, #2 and #3) in order.

> **Why a separate Python environment?** Colab's default Python version is now newer than the versions supported by AiZynthFinder (Python 3.10 to 3.12), so a plain `pip install aizynthfinder` fails with *"No matching distribution found"*. This notebook therefore installs AiZynthFinder into its own Python 3.11 environment and runs the searches there, while the input box and results are displayed in the notebook itself. This also protects the notebook from future changes to Colab's default Python.

---

###Attribution & Licensing
Article License: The original article is licensed under a Creative Commons Attribution 4.0 International License.

Data Waiver: The data from the original article is provided under the Creative Commons Public Domain Dedication waiver.

Software License: The original AiZynthFinder software is distributed under the MIT License.

Note: Modifications have been made to the original workflow to adapt it for the Google Colab environment.

###References
Genheden, S., Thakkar, A., Chadimová, V. et al. AiZynthFinder: a fast, robust and flexible open-source software for retrosynthetic planning. J Cheminform 12, 70 (2020). https://doi.org/10.1186/s13321-020-00472-1


# AiZynthFinder-based Retrosynthesis Tool: Google Colab Notebook

**Author:** Dr. Sunil Paliwal, Stevens Institute of Technology  
**License:** [MIT License](https://opensource.org/licenses/MIT)  
**Description:** A notebook for retrosynthetic planning using Monte Carlo tree search, based on the open-source software AiZynthFinder, as described in the paper: *Genheden et al., J. Cheminform (2020)*.

Paper link: https://doi.org/10.1186/s13321-020-00472-1

Github link: https://github.com/MolecularAI/aizynthfinder

# Interactive Retrosynthesis Setup: Navigating the implementation code:

This google colab notebook provides a clean, step-by-step implementation guide to help you move from a SMILES string to a fully visualized retrosynthetic tree using the AiZynthFinder API.

This notebook sets up the `aizynthfinder` environment (in its own Python 3.11 environment, because AiZynthFinder does not yet support Colab's default Python), downloads the required USPTO models and ZINC stock collections, and launches an interactive interface for retrosynthetic analysis.

---
### Getting Started

Run the three code cells below **in order**. If your Colab session disconnects, re-run all three.


In [ ]:
# 1. Install AiZynthFinder and dependencies
# (installed in a separate Python 3.11 environment)
import os
os.environ.pop("UV_SYSTEM_PYTHON", None)

!pip install -q uv
!uv venv --clear --python 3.11 /content/azf_env
!uv pip install -q --python /content/azf_env/bin/python "aizynthfinder==4.4.1"
print('Installation complete.')


In [ ]:
# 2. Download public USPTO models and ZINC stock collection
# This will also generate the 'config.yml' file automatically
!/content/azf_env/bin/download_public_data /content
print('Data download and configuration complete.')


In [ ]:
# 3. Launch the Interactive GUI
# Usage: Enter a SMILES string and click 'Run Search'
import subprocess, shutil, os, json, glob, re
from pathlib import Path
import ipywidgets as w
from IPython.display import display, Image, clear_output

# Helper script that runs the AiZynthFinder search inside the Python 3.11 environment
Path("/content/azf_worker.py").write_text('''
import sys, json
from aizynthfinder.aizynthfinder import AiZynthFinder

smiles, outdir = sys.argv[1], sys.argv[2]
finder = AiZynthFinder(configfile="/content/config.yml")
finder.stock.select("zinc")
finder.expansion_policy.select("uspto")
finder.filter_policy.select("uspto")
finder.target_smiles = smiles
finder.tree_search()
finder.build_routes()
stats = finder.extract_statistics()
for i, img in enumerate(finder.routes.make_images()[:10]):
    if img is not None:
        img.save(f"{outdir}/route_{i+1}.png")
json.dump(stats, open(f"{outdir}/stats.json", "w"), default=str)
''')

smiles_box = w.Text(value="CC(=O)Oc1ccccc1C(=O)O", description="Target SMILES",
                    layout=w.Layout(width="650px"), style={"description_width": "110px"})
run_button = w.Button(description="Run Search", button_style="primary")
output = w.Output()

def run_search(_):
    with output:
        clear_output()
        print("Searching... this can take a minute.")
        shutil.rmtree("/content/azf_out", ignore_errors=True)
        os.makedirs("/content/azf_out")
        result = subprocess.run(
            ["/content/azf_env/bin/python", "/content/azf_worker.py",
             smiles_box.value.strip(), "/content/azf_out"],
            capture_output=True, text=True, cwd="/content")
        clear_output()
        if result.returncode != 0:
            print("Search failed:\n", result.stderr[-3000:])
            return
        stats = json.load(open("/content/azf_out/stats.json"))
        for key, value in stats.items():
            print(f"{key}: {value}")
        files = sorted(glob.glob("/content/azf_out/route_*.png"),
                       key=lambda f: int(re.findall(r"\d+", os.path.basename(f))[0]))
        for f in files:
            print(f"\nRoute {os.path.basename(f)[6:-4]}")
            display(Image(f))

run_button.on_click(run_search)
display(smiles_box, run_button, output)


### Instructions for using the interactive search interface:

1.  **Locate the interface:** The interface is displayed directly below the last code cell (Cell #3). It consists of a text box labeled "Target SMILES", a blue "Run Search" button, and an area below them where results will appear.
2.  **Input SMILES String:** In the "Target SMILES" field, enter the simplified molecular-input line-entry system (SMILES) string for the molecule you want to retrosynthesize.
3.  **Example SMILES:** Let's use **Aspirin** as an example. Its SMILES string is: `CC(=O)Oc1ccccc1C(=O)O` (this is the default value in the box).
4.  **Run Search:** Click the "Run Search" button. The search can take a minute or so. To analyze a different molecule, replace the SMILES string and click "Run Search" again.


 5.  **Interpret Results:** When the search finishes, the interface prints a list of search statistics, followed by images of the predicted reaction pathways (up to the top 10), from the target molecule back to precursors.
6. **Route 1 (and other numbered routes):** AiZynthFinder often identifies multiple potential pathways or 'solutions' to synthesize your target molecule. Each numbered route (Route 1, Route 2, etc.) is a distinct, complete retrosynthetic pathway from the target molecule back to a set of presumed starting materials. Routes are listed in order of their score, so Route 1 is the top-ranked pathway.
 7. **Search statistics:** The lines printed above the route images summarize the search, for example whether a solution was found (`is_solved`), how many routes were found (`number_of_routes`), how many of them are fully solved (`number_of_solved_routes`), and the `top_score` of the best route. The score is a metric used by AiZynthFinder to evaluate the 'goodness' or 'quality' of a route. Generally, a higher score suggests a more promising route, for example one whose precursors are known, readily available, or synthetically accessible compounds.
 8. **Troubleshooting:** If a search fails, the error message is printed in the output area below the button. If the notebook stops working after a session disconnect, re-run Cells #1, #2 and #3 in order.

